# Circuit Discovery

**Survey Reference:** Section 8.2 - Circuit Discovery Methods

## Overview

**Circuits** are minimal subnetworks that implement specific behaviors. Circuit discovery aims to identify which components (attention heads, MLP neurons, features) are responsible for a particular computation.

### Key Methods

1. **Activation Patching:** Replace activations to measure importance
2. **Path Patching:** Trace information flow through specific paths
3. **Attribution Patching:** Efficient gradient-based approximation
4. **Automated Circuit Discovery:** Search algorithms for circuits

## Learning Objectives

1. Understand activation patching and its variants
2. Implement path patching for circuit tracing
3. Use attribution methods for scalable discovery
4. Identify circuits for specific behaviors

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable, Optional
from tqdm import tqdm

# TransformerLens is ideal for circuit analysis
# from transformer_lens import HookedTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Activation Patching

### The Core Idea

Given two inputs:
- **Clean input:** Produces correct behavior
- **Corrupted input:** Produces incorrect behavior

We patch activations from clean → corrupted (or vice versa) to measure each component's importance.

$$\text{Effect}(c) = \text{Metric}(\text{model with patch}) - \text{Metric}(\text{corrupted baseline})$$

In [ ]:
def activation_patching(
    model,
    clean_input,
    corrupted_input,
    metric_fn: Callable,
    patch_locations: list[str],
):
    """
    Perform activation patching to identify important components.
    
    Args:
        model: The model to analyze
        clean_input: Input that produces correct behavior
        corrupted_input: Input that produces incorrect behavior
        metric_fn: Function to evaluate behavior (e.g., logit difference)
        patch_locations: List of activation names to patch
        
    Returns:
        effects: Dict mapping location -> patching effect
    """
    effects = {}
    
    # Get clean activations
    # clean_cache = run_with_cache(model, clean_input)
    
    # Get corrupted baseline
    # corrupted_metric = metric_fn(model(corrupted_input))
    
    # For each location, patch and measure effect
    for loc in patch_locations:
        # TODO: Implement patching logic
        # patched_output = run_with_patch(model, corrupted_input, loc, clean_cache[loc])
        # effects[loc] = metric_fn(patched_output) - corrupted_metric
        pass
    
    return effects

## 2. Path Patching

Path patching traces information flow through specific paths:

1. Patch **sender** component with clean activation
2. Keep **receiver** component's other inputs corrupted
3. Measure if the path carries important information

This reveals **how** information flows, not just **where**.

In [ ]:
def path_patching(
    model,
    clean_input,
    corrupted_input,
    metric_fn: Callable,
    sender: str,
    receiver: str,
):
    """
    Measure importance of a specific path from sender to receiver.
    
    Args:
        sender: Name of the sending component
        receiver: Name of the receiving component
        
    Returns:
        path_effect: How much this path contributes to the behavior
    """
    # TODO: Implement path patching
    # 1. Run corrupted input, save all activations
    # 2. Run clean input, save sender output
    # 3. Re-run corrupted but patch sender→receiver edge with clean
    # 4. Measure effect
    pass

## 3. Attribution Patching (Gradient-Based)

Full patching is expensive ($O(n)$ forward passes). Attribution patching approximates patching effects using gradients:

$$\text{AttrPatch}(c) \approx \nabla_{a_c} \text{Metric} \cdot (a_c^{\text{clean}} - a_c^{\text{corrupted}})$$

This needs only **2** forward passes + **1** backward pass.

In [ ]:
def attribution_patching(
    model,
    clean_input,
    corrupted_input,
    metric_fn: Callable,
):
    """
    Gradient-based approximation to activation patching.
    
    Much faster than full patching for initial screening.
    """
    # Get clean activations (with gradients)
    # clean_output, clean_acts = model.run_with_cache(clean_input)
    
    # Get corrupted activations
    # corrupted_output, corrupted_acts = model.run_with_cache(corrupted_input)
    
    # Compute gradients of metric w.r.t. corrupted activations
    # metric = metric_fn(corrupted_output)
    # metric.backward()
    
    # Approximate effect: grad * (clean - corrupted)
    # effects = {}
    # for name, act in corrupted_acts.items():
    #     delta = clean_acts[name] - act
    #     effect = (act.grad * delta).sum()
    #     effects[name] = effect.item()
    
    # return effects
    pass

## 4. Example: Indirect Object Identification (IOI)

A classic circuit analysis task:

> "When Mary and John went to the store, John gave a drink to **___**"

The model should predict "Mary" (the indirect object, not the subject).

In [ ]:
def create_ioi_examples():
    """
    Create clean and corrupted IOI examples.
    """
    clean_prompts = [
        "When Mary and John went to the store, John gave a drink to",
        "When Alice and Bob went to the park, Bob gave a flower to",
    ]
    
    # Corrupted: swap names so model should predict wrong name
    corrupted_prompts = [
        "When John and John went to the store, John gave a drink to",
        "When Bob and Bob went to the park, Bob gave a flower to",
    ]
    
    # Expected answers
    correct_answers = ["Mary", "Alice"]
    incorrect_answers = ["John", "Bob"]
    
    return clean_prompts, corrupted_prompts, correct_answers, incorrect_answers

def ioi_metric(logits, correct_token_id, incorrect_token_id):
    """
    Metric for IOI: logit difference between correct and incorrect answer.
    """
    return logits[..., correct_token_id] - logits[..., incorrect_token_id]

## 5. Visualizing Circuit Structure

After patching, visualize which components are important.

In [ ]:
def visualize_patching_results(effects: dict, n_layers: int, n_heads: int):
    """
    Visualize activation patching results as a heatmap.
    """
    # Create matrix for attention heads
    head_effects = np.zeros((n_layers, n_heads))
    
    for name, effect in effects.items():
        if 'attn' in name:
            # Parse layer and head from name
            # head_effects[layer, head] = effect
            pass
    
    plt.figure(figsize=(12, 8))
    plt.imshow(head_effects, cmap='RdBu_r', aspect='auto')
    plt.colorbar(label='Patching Effect')
    plt.xlabel('Head')
    plt.ylabel('Layer')
    plt.title('Attention Head Importance')
    plt.show()

# TODO: Apply to actual patching results

## 6. Automated Circuit Discovery

Recent work automates circuit discovery:

- **ACDC (Automatic Circuit Discovery):** Iteratively prune unimportant edges
- **Edge Attribution Patching:** Efficient path-level attribution
- **Subnetwork Probing:** Learn circuit masks end-to-end

In [ ]:
def acdc_circuit_discovery(
    model,
    clean_input,
    corrupted_input,
    metric_fn: Callable,
    threshold: float = 0.01,
):
    """
    Simplified ACDC-style circuit discovery.
    
    Iteratively removes edges with small effects.
    """
    # Start with all edges
    # active_edges = get_all_edges(model)
    
    # Iteratively remove edges with effect < threshold
    # while True:
    #     effects = compute_edge_effects(model, active_edges, ...)
    #     min_effect_edge = min(effects, key=effects.get)
    #     if effects[min_effect_edge] < threshold:
    #         active_edges.remove(min_effect_edge)
    #     else:
    #         break
    
    # return active_edges
    pass

## Key Takeaways

1. **Activation patching** measures component importance by intervention
2. **Path patching** traces information flow between components
3. **Attribution patching** provides fast gradient-based approximations
4. **Clean/corrupted pairs** are essential for circuit analysis
5. **Visualization** helps understand circuit structure

## Next Steps

→ **02_causal_verification.ipynb**: Verify that discovered circuits actually implement the behavior